# 124. Binary Tree Maximum Path Sum
**Difficulty:** 🔴 Hard · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/binary-tree-maximum-path-sum/

## 💡 Concepts

**Core concept(s):** DFS that returns a node's best **downward** contribution while tracking the best **full** path globally.

**Why it applies here:** A path can bend at a node: come up the left, pass through the node, go down the right. So at each node we (a) update a global best using node + best-left-down + best-right-down, and (b) return only the single best branch upward (a path can't split when handed to a parent).

**Key intuition:** At each node, the best path *through* it uses both downward arms; but you can only *hand your parent* one arm.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- Recursion that both returns a value and updates a shared best.

## 📝 Problem

A path is any sequence of connected nodes; it doesn't have to pass through the root. Return the largest possible sum of node values along a path.

**Example**
```
[-10,9,20,None,None,15,7] -> 42   (15 -> 20 -> 7)
[1,2,3] -> 6                        (2 -> 1 -> 3)
```

> Two approaches: naive `O(n²)` and one-pass DFS `O(n)`.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Recompute Downward Sums (worst)

**Idea:** For each node compute its best path *through* it by measuring the best downward path on each side — but recomputing those downward sums at every node repeats work.

**Time:** `O(n²)`.

**Space:** `O(h)`.

In [ ]:
def max_path_brute(root: Optional[TreeNode]) -> int:
    def down(n):                           # best sum of a path going straight DOWN from n
        if not n:
            return 0
        return max(0, n.val + max(down(n.left), down(n.right)))  # 0 = skip a harmful branch
    best = [float("-inf")]                 # global best (in a list so inner functions can update it)
    def visit(n):
        if not n:
            return
        # Best path that BENDS through n = n + best downward-left + best downward-right.
        through = n.val + max(0, down(n.left)) + max(0, down(n.right))
        best[0] = max(best[0], through)
        visit(n.left); visit(n.right)      # (down() is recomputed at each node -> O(n^2))
    visit(root)
    return best[0]

### Approach 2 — One-Pass DFS (optimal)

**Idea:** A single DFS returns each node's best downward arm and, along the way, updates the global best with the full bent path (node + both arms). Negative arms are clamped to 0 (skip them).

**Time:** `O(n)`.

**Space:** `O(h)`.

In [ ]:
def max_path_optimal(root: Optional[TreeNode]) -> int:
    best = [float("-inf")]                 # global best answer found anywhere
    def gain(n):                           # returns the best single downward arm from n
        if not n:
            return 0
        left = max(0, gain(n.left))        # ignore a branch that would hurt the sum (clamp to 0)
        right = max(0, gain(n.right))
        best[0] = max(best[0], n.val + left + right)  # a path bending through n uses BOTH arms
        return n.val + max(left, right)    # but we can only hand our PARENT one arm
    gain(root)
    return best[0]

In [ ]:
# Correctness check
tests = [([-10,9,20,None,None,15,7],42), ([1,2,3],6), ([-3],-3), ([2,-1],2)]
for vals, exp in tests:
    root = build_tree(vals)
    a, b = max_path_brute(root), max_path_optimal(root)
    print(f"{vals} -> brute={a}, optimal={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n),)
solutions = {
    "brute   O(n^2)": max_path_brute,
    "optimal O(n)  ": max_path_optimal,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Return one thing, track another:** the DFS *returns* the best single arm but *updates* a global best with the full bent path — a very common tree trick.
- **Clamp negatives to zero:** a branch that hurts the sum is simply not taken.
- **Signal:** "maximum path sum", "best path that may bend", "diameter-like" tree questions.
- **Related problems:** Diameter of Binary Tree, House Robber III, Longest Univalue Path.
- **Common pitfalls:** (1) returning the bent (two-arm) value to the parent — a parent's path can't split; (2) forgetting all-negative trees (seed best with -inf).